In [1]:
# 1. Import Libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 2. Generate Synthetic Sensor Time-Series Data
np.random.seed(42)
hours = 5000
time_index = pd.date_range(start='2026-01-01', periods=hours, freq='H')

In [3]:
# Simulating normal operating conditions with occasional spikes
voltage = np.random.normal(230, 5, hours) 
current = np.random.normal(10, 1.5, hours)
temperature = np.random.normal(45, 5, hours)
vibration = np.random.normal(2, 0.5, hours)

In [4]:
df = pd.DataFrame({'timestamp': time_index, 'voltage': voltage, 'current': current, 
                   'temperature': temperature, 'vibration': vibration})

In [5]:
# Introduce failure conditions (if temp > 55 and vibration > 3, failure happens in next 24h)
df['failure_imminent'] = 0
for i in range(len(df) - 24):
    if df.loc[i, 'temperature'] > 55 and df.loc[i, 'vibration'] > 3.0:
        df.loc[i+1:i+24, 'failure_imminent'] = 1

In [6]:
# 3. Advanced Feature Engineering
df = df.sort_values('timestamp').reset_index(drop=True)

In [7]:
# Rolling Averages (Smoothing out noise)
df['temp_roll_mean_6h'] = df['temperature'].rolling(window=6).mean()
df['vib_roll_mean_6h'] = df['vibration'].rolling(window=6).mean()

# Rolling Standard Deviation (Capturing volatility/frequency domain proxy)
df['current_roll_std_12h'] = df['current'].rolling(window=12).std()
df['vib_roll_std_12h'] = df['vibration'].rolling(window=12).std()

In [8]:
# Lag Features (What was the value 1 hour ago?)
df['voltage_lag_1h'] = df['voltage'].shift(1)
df['temp_lag_1h'] = df['temperature'].shift(1)

# Drop NaN values created by rolling/lag operations
df = df.dropna()

In [9]:
# 4. Train-Test Split
features = ['voltage', 'current', 'temperature', 'vibration', 
            'temp_roll_mean_6h', 'vib_roll_mean_6h', 'current_roll_std_12h', 
            'vib_roll_std_12h', 'voltage_lag_1h', 'temp_lag_1h']

In [10]:
X = df[features]
y = df['failure_imminent']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
# 5. Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, random_state=42)

In [12]:
# 6. Evaluation
y_pred = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9899799599198397

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99       988
           1       0.00      0.00      0.00        10

    accuracy                           0.99       998
   macro avg       0.49      0.50      0.50       998
weighted avg       0.98      0.99      0.98       998



In [13]:
# 7. Export the Model and Feature List
joblib.dump(rf_model, 'rf_maintenance_model.pkl')
joblib.dump(features, 'model_features.pkl')
print("Model successfully saved as rf_maintenance_model.pkl")

Model successfully saved as rf_maintenance_model.pkl
